# 14 · Grouped, paired, seed-separated statistical analysis

**OncoPlate v3.0.0 — implementation of the accepted v3.0 plan**

Requires fixed predictions and independently adjudicated outcomes from notebook 13.

This is research software, not a validated cancer-risk or chemical-detection product. Run cells in order. Missing independently collected data or approval records are genuine prerequisites, not permission to substitute synthetic results.

In [ ]:
from pathlib import Path
import os, sys, json
ON_COLAB = "google.colab" in sys.modules or bool(os.environ.get("COLAB_RELEASE_TAG"))
if ON_COLAB and not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")
ROOT = Path(os.environ.get("ONCOPLATE_DRIVE_ROOT", "/content/drive/MyDrive/OncoPlate_Research"))
pointer = ROOT / ".oncoplate_install.json"
preferred = json.loads(pointer.read_text())["repository_path"] if pointer.exists() else str(ROOT / "oncoplate-research")
REPO = Path(os.environ.get("ONCOPLATE_REPO", preferred))
if not (REPO / "src/oncoplate").exists():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "src/oncoplate").exists(): REPO = candidate; break
assert (REPO / "src/oncoplate").exists(), "Run the supplied installer notebook or set ONCOPLATE_REPO to the extracted repository."
sys.path.insert(0, str(REPO / "src"))
from oncoplate.config import load_config, paths, initialize
from oncoplate.io import read_json, write_json, read_table, write_table, read_jsonl, write_jsonl, utcnow
cfg = load_config(REPO, root=ROOT, mode=os.environ.get("ONCOPLATE_MODE", "research"))
p = paths(cfg)
print("Dataset:", cfg["study"]["dataset"], "| Mode:", cfg["mode"], "| Persistent root:", cfg["root"])


## 1. Load and validate the matched cohort

In [ ]:
from oncoplate.governance import verify_lock
from oncoplate.statistics import validate_paired_results,paired_bootstrap,risk_coverage
from oncoplate.evaluation import evaluate_results
lock=verify_lock(p['private']/'analysis_lock.json');protocol=lock['protocol']
results=read_table(p['private']/'test_evaluation_results.csv');results['seed']=results.seed.astype(int)
validate_paired_results(results,'M7','M3',protocol['model_seeds'])
print(results.groupby(['method','seed']).record_id.nunique())

## 2. Compute exact registered endpoints

In [ ]:
table=evaluate_results(results,protocol['domain_mixture'])
write_table(p['reports']/'main_selective_results.csv',table);display(table)
summary,replicates=paired_bootstrap(results,'M7','M3',mixture=protocol['domain_mixture'],seeds=protocol['model_seeds'],B=protocol['bootstrap_replicates'],coverage_margin=protocol['coverage_margin'])
write_json(p['reports']/'primary_paired_comparison.json',summary)
from oncoplate.io import atomic_npz
atomic_npz(p['private']/'primary_bootstrap_replicates.npz',differences=replicates)
print(json.dumps(summary,indent=2))

## 3. Show descriptive risk–coverage curves
M3 has no source gate; M7 does. Never interpolate beyond a policy’s feasible coverage.

In [ ]:
from oncoplate.reporting import plot_risk_coverage
from oncoplate.statistics import bool_series
curves={}
for method in ['M3','M7']:
    part=results[(results.method==method)&(results.seed==0)].copy()
    part['curve_eligible']=bool_series(part.informative)
    if method=='M7':part['curve_eligible'] &= bool_series(part.eligible)
    curve=risk_coverage(part,eligible_col='curve_eligible',mixture=protocol['domain_mixture'])
    write_table(p['reports']/f'{method}_seed0_descriptive_curve.csv',curve);curves[method]=curve
plot_risk_coverage(curves,p['reports']/'primary_risk_coverage.png')

## 4. Preserve null or unstable results

In [ ]:
print('Joint success rule:',summary['success'])
print('The interval must favour M7 on risk AND pass the coverage constraint. No single seed is selected as the headline.')
print('Seed-specific results are in the table; bootstrap samples independent groups, not seeds or photographs.')

## Completion and resumption
Outputs are written to the displayed persistent dataset-specific directories. Keep raw inputs, reviewer records, arrays and checkpoints private. Re-running a completed fit verifies its configuration rather than silently changing it. Use notebook 99 only after reviewing aggregate results; nothing here pushes to GitHub automatically.
